# RSI Mean-Reversion Significance Analysis
This notebook applies prior-day RSI positions and robust time-series inference. Run it from the repository root after configuring `DB_URL`.

In [ ]:
import numpy as np
import pandas as pd
from sqlalchemy import text
from database import create_db_engine
from statistics_analysis import hac_mean_test, moving_block_bootstrap_mean

engine = create_db_engine()

In [ ]:
query = text("""
SELECT p.stock_id, p.date, p.close, i.rsi_14
FROM daily_prices p
JOIN indicators i ON p.stock_id = i.stock_id AND p.date = i.date
ORDER BY p.stock_id, p.date
""")
data = pd.read_sql(query, engine)
data['close'] = data['close'].astype(float)
grouped = data.groupby('stock_id')
data['market_return'] = grouped['close'].pct_change()
previous_rsi = grouped['rsi_14'].shift(1)
raw = pd.Series(np.nan, index=data.index)
raw[previous_rsi < 30] = 1.0
raw[previous_rsi > 70] = 0.0
data['position'] = raw.groupby(data['stock_id']).ffill().fillna(0.0)
data['strategy_return'] = data['position'] * data['market_return']
data['difference'] = data['strategy_return'] - data['market_return']
daily_difference = data.groupby('date')['difference'].mean().dropna()

In [ ]:
hac_result = hac_mean_test(daily_difference)
bootstrap_result = moving_block_bootstrap_mean(daily_difference, block_size=20)
print('HAC mean test:', hac_result)
print('Moving-block bootstrap:', bootstrap_result)